In [22]:
from openai import OpenAI
import os
token = ''
client = OpenAI(api_key=token)

import pymupdf
import os


def summarization_gen(instruction, model_temp, model_name):
    response = client.chat.completions.create(
        model=model_name,  # Or the specific model you have access to
        messages=[
            {"role": "system", "content": "You are a scientific assistant specializing in molecular biology and bioinformatics. Your goal is to generate concise, structured gene descriptions based on extracted scientific data."},
            {"role": "user", "content": f"{instruction}"}
        ],
    )
    
    # Extract the response
    # print(response)
    information = response.choices[0].message.content.strip()
    
    return information



def create_prompt_from_text(extracted_info_json):
    instruction = f"""
    ## **Task Description**
    Generate a concise, well-structured gene description using the extracted information below. Follow the structure of the example provided. Summarize multiple terms into high-level categories to avoid excessive listing.

    ## **Example Gene Description**:
    Exhibits protein serine/threonine kinase activity. Involved in several processes, including male gonad development; positive regulation of brood size; and positive regulation of cell cycle process. Predicted to be part of cyclin-dependent protein kinase holoenzyme complex. Predicted to be active in cytoplasm and nucleus. Is expressed in gonad; intestine; neurons; and vulva. Used to study cancer. Human ortholog(s) of this gene implicated in several diseases, including familial melanoma; high-grade glioma (multiple); pancreatic carcinoma (multiple); primary autosomal recessive microcephaly 12; and type 2 diabetes mellitus. Orthologous to human CDK4 (cyclin-dependent kinase 4) and CDK6 (cyclin-dependent kinase 6). Decreased activity of SEL-12 ameliorates the lethality associated with APL-1 overexpression.

    ## **Sentence Templates for Gene Description**:
    - **Molecular Function**: Exhibits <list of gene ontology molecular function terms>.
    - **Biological Process**: Involved in **broad biological categories** such as <high-level biological processes>.
    - **Subcellular Localization**: Localizes to <list of gene ontology cellular component terms>.
    - **Pathways**: Functions in <list of pathway terms>.
    - **Disease Relevance**: Used to study <list of disease ontology terms>.
    - **Expression**: Expressed in <list of anatomy ontology terms>.
    - **Implication of Human Orthologs in disease**: Human ortholog(s) of this gene implicated in <list of disease ontology terms>.
    - **Human Orthologs**: Orthologous to <list of human orthologs>.
    - **Modifiers**: <Modifier> ameliorates or exacerbates the disease phenotype or condition.

    ## **Rules for Summarization**
    - **General Guidelines**:
      - Maintain **scientific tone**, avoiding redundant phrases.
      - Ensure **grammatical correctness**.
      - Use **semicolons (;)** for separating multiple terms in a category.
      - **Omit** fields with no available data.

    ## **Extracted Information**:
    {extracted_info_json}

    Using this structured information, generate a complete and well-formatted gene description.
    """

    return instruction

In [25]:
import os
import json
from openai import OpenAI
from pprint import pprint

with open('data_pairs_20250510.json', 'r') as f:
    gene_data = json.load(f)

models = ['gpt-4-turbo', 'o1', 'gpt-4.5-preview', 'gpt-4o', 'gpt-4o-mini']
folder = '/Users/pengyuan/Desktop/GeneSum/alliance_files/gene_descriptions/no_desc'

for gene_name in gene_data.keys():
    if not gene_data[gene_name]['gt']['description'] or len(gene_data[gene_name]['gt']['description'])<5:
        print(gene_name)
        paper_info = {}
        paper_info['gene_name'] = gene_name
        for key in gene_data[gene_name]['paper_inf'].keys():
            if len(gene_data[gene_name]['paper_inf'][key])>0:
                print(gene_data[gene_name]['paper_inf'][key])
                for item in gene_data[gene_name]['paper_inf'][key]:
                    paper_info[key] = [item['sentence template'] for item in gene_data[gene_name]['paper_inf'][key] if 'sentence template' in item.keys()]
        instruction = create_prompt_from_text(paper_info)
        print(instruction) 
        model_temp = 0.0
        
        gene_info = {}
        for model_name in models:
            gene_info[model_name] = summarization_gen(instruction, model_temp, model_name)
        print('*'*10)
        print(gene_info)
        print('*'*10)
        task_name = gene_name+ '_NoDesc_paperinfo_20250510.json'
        with open(os.path.join(folder, task_name), 'w') as f:
            json.dump(gene_info, f, indent=4)

ZC239.1
[{'sentence template': 'zc239.1 is involved in the inflammatory response.', 'evidence_sentences': "['In the worms treated with EC, a significant descent was produced in the expression of ZC239.12, a gene homolog of human TNFAIP1 associated with inflammation and apoptosis…”']", 'references': 'PMID_38247503'}]
[{'sentence template': 'zc239.1 is part of the tumor necrosis factor-mediated signaling pathway.', 'evidence_sentences': "['ZC239.12 was analyzed as an ortholog of human TNFAIP1 (TNF alpha-induced protein 1) … TNFAIP1 mRNA levels increased robustly in the transgenic CL4176 strain … This gene can be induced by TNFα.”']", 'references': 'PMID_38247503'}]
[{'sentence template': 'Epicatechin ameliorates the Aβ-induced toxicity phenotype.', 'evidence_sentences': "['All in all, the observed decrease in TNFAIP1 expression induced by EC can be important to explain its apparent neuroprotective effects, as TNFAIP1 could be a crucial pathological biomarker of AD and may play an importa